<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/LoRA-Plus/blob/main/ML_Project_LoRA%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Global Library Installations and Drive Imports



In [ ]:
import os

REPO_URL = "https://github.com/Ramprasath-Meganathan/LoRA-Plus.git"
REPO_DIR = "/content/LoRA-Plus"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
# INSTALL/UPGRADE ONLY THE LIBRARIES WHAT COLAB DOESN'T SHIP WITH.
!pip install -q -U peft evaluate
#!pip uninstall -y -q torchao

In [ ]:
# MOUNT DRIVE AND IMPORT PATHS
import os, json, time, glob, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, datasets, peft, evaluate
import gc
from google.colab import drive

#Mount drive once only
drive.mount('/content/drive')

from datasets import load_dataset
from torch.optim import AdamW
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

#One root folder for everything this notebook saves
ROOT = "/content/drive/MyDrive/LoRA_Plus_Project"
os.makedirs(f"{ROOT}/runs", exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#ENVIRONMENT LOG TO CHECK THE LIBRARY VERSIONS
ENV = {
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "peft": peft.__version__,
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "python": platform.python_version(),
}
print(ENV)
json.dump(ENV, open(f"{ROOT}/env.json", "w"), indent=2)

{'torch': '2.14.1+cu130', 'transformers': '5.22.1', 'datasets': '4.8.5', 'peft': '0.21.2', 'gpu': 'Tesla T4', 'python': '3.13.16'}


In [ ]:
# # Mount Google Drive for saving checkpoints to protect against session timeouts
# drive.mount('/content/drive')
# CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_Plus_Checkpoints/"
# if os.path.exists('/content/drive/MyDrive'):
#   print("Google drive is already verified. Skipping mount request")
# else:
#   print("Target mount point not detected. Mounting Google Drive...")
#   drive.mount('/content/drive')

# os.makedirs(CHECKPOINT_DIR, exist_ok=True)
# print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google drive is already verified. Skipping mount request
Checkpoints will be saved to: /content/drive/MyDrive/LoRA_Plus_Checkpoints/


In [ ]:
#EXPERIMENT SETTINGS
DATASET = "sst2"  # change to "qqp" when you move to QQP
BS = 32# setting as per the paper
SEED_SCREEN = 42 # screening uses 1 seed
FINAL_SEEDS = [42,43,44] # same seeds for LoRA and LoRA+
RESULTS_CSV = f"{ROOT}/results_{DATASET}.csv"   # one CSV per dataset

In [ ]:
# LOAD AND TOKENIZE SST-2
dataset = load_dataset("nyu-mll/glue", "sst2")
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
    # max_length=128 is set to match the paper.
    # No padding here: the data collator pads each batch dynamically.
  return tokenizer(
      batch["sentence"],
      truncation=True,
      max_length=128
  )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence", "idx"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)

tokenized_dataset.set_format("torch")
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})


In [ ]:
# Toggle this to False when you are ready for the full 67k run!
PILOT_RUN = False

if PILOT_RUN:
    # Subset the tokenized datasets safely
    pilot_train_dataset = tokenized_dataset["train"].select(range(1000))
    pilot_eval_dataset = tokenized_dataset["validation"].select(range(min(200, len(tokenized_dataset["validation"]))))
    print(f"Pilot Mode Active: Subsampled Train ({len(pilot_train_dataset)} examples) and Eval ({len(pilot_eval_dataset)} examples).")
else:
    pilot_train_dataset = tokenized_dataset["train"]
    pilot_eval_dataset = tokenized_dataset["validation"]
    print(f" Full Mode Active: Training on all {len(pilot_train_dataset)} examples.")

 Full Mode Active: Training on all 67349 examples.


In [ ]:
# Same config for LoRA and LoRA+, so the only difference is the learning rates.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,                       # paper: r = alpha = 8
    lora_dropout=0.1,                   # dropout value
    target_modules=["query", "value"],  # RoBERTa modules used in the paper
    modules_to_save=["classifier"],     # classification head
    bias="none",
)

In [ ]:
# Evaluation Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_prediction):
    logits, labels = eval_prediction
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],
        "f1_macro": f1_metric.compute(
            predictions=predictions,
            references=labels,
            average="macro"
        )["f1"]
    }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
# Experiment Function
def run_roberta(lam, seed, epochs, base_lr=1e-4, bs=BS,
                stage="screening", max_steps=-1):
    """lam=1 is standard LoRA; lam>1 is LoRA+ (B's lr = lam * A's lr).
    stage: "test" (not saved to CSV), "screening", or "final"."""
    run_name = f"roberta_{DATASET}_lam{lam}_lr{base_lr}_seed{seed}_ep{epochs}_{stage}"
    json_path = f"{ROOT}/runs/{run_name}.json"

    # If Colab disconnected mid-loop, finished runs are skipped on rerun.
    if stage != "test" and os.path.exists(json_path):
        print("Already done, skipping:", run_name)
        return json.load(open(json_path))

    set_seed(seed)  # same seed = same LoRA-A init, head init, and data order
    base = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
    m = get_peft_model(base, lora_config)

    # Split trainable params into the three LoRA+ groups.
    A, B, O = [], [], []
    for n, p in m.named_parameters():
        if not p.requires_grad:
            continue
        (A if "lora_A" in n else B if "lora_B" in n else O).append(p)

    opt = AdamW(
        [{"params": A, "lr": base_lr},            # matrix A
         {"params": B, "lr": base_lr * lam},      # matrix B: lam x larger
         {"params": O, "lr": 1e-4}],           # classifier head
        betas=(0.9, 0.99), eps=1e-8, weight_decay=0.01,   # paper's AdamW settings
    )
    for name, g in zip(["A", "B", "other"], opt.param_groups):
        print(name, sum(p.numel() for p in g["params"]), g["lr"])  # sanity check

    args = TrainingArguments(
        output_dir=f"{ROOT}/tmp/{run_name}",
        seed=seed, data_seed=seed,
        num_train_epochs=epochs,
        max_steps=max_steps,
        metric_for_best_model = "accuracy",
        per_device_train_batch_size=bs,
        per_device_eval_batch_size=64,
        eval_strategy="steps", eval_steps=200,   # points for convergence curves
        save_strategy="no",                      # no checkpoints; results go to JSON
        load_best_model_at_end=False,            # report the final model, not the best epoch
        logging_steps=50,
        fp16=torch.cuda.is_available(),
        report_to="none",
        # Note: no learning_rate here; the lrs come from our optimizer groups.
    )

    tr = Trainer(
        model=m, args=args,
        train_dataset=pilot_train_dataset, eval_dataset=pilot_eval_dataset,
        processing_class=tokenizer, data_collator=data_collator,
        compute_metrics=compute_metrics,
        optimizers=(opt, None),   # Trainer adds the linear schedule around our optimizer
    )
    if torch.cuda.is_available():
      torch.cuda.reset_peak_memory_stats()
    res = tr.train()
    final = tr.evaluate()

    log = tr.state.log_history
    evals = [l for l in log if "eval_accuracy" in l]
    trains = [l for l in log if "loss" in l]

    row = {
        "stage": stage, "model": "roberta-base", "dataset": DATASET,
        "method": "LoRA" if lam == 1 else "LoRA+",
        "lam": lam, "eta_A": base_lr, "eta_B": base_lr * lam,
        "seed": seed, "epochs": epochs, "batch_size": bs,
        "final_acc": final["eval_accuracy"],
        "best_acc": max([e["eval_accuracy"] for e in evals] + [final["eval_accuracy"]]),
        "best_f1_macro": max([e["eval_f1_macro"] for e in evals] + [final["eval_f1_macro"]]),
        "final_eval_loss": final["eval_loss"],
        "last_train_loss": trains[-1]["loss"] if trains else None,
        "train_runtime_s": res.metrics["train_runtime"],   # includes periodic evals
        "samples_per_s": res.metrics["train_samples_per_second"],
        "total_steps": res.global_step,
        "peak_mem_gb": (torch.cuda.max_memory_allocated() / 1e9
                         if torch.cuda.is_available() else None),
        "trainable_params": sum(p.numel() for p in m.parameters() if p.requires_grad),
        "torch": ENV["torch"], "transformers": ENV["transformers"], "peft": ENV["peft"],
        "final_f1_macro": final["eval_f1_macro"],
    }

    if stage != "test":
        # Full curves go in the JSON; the summary row is appended to the CSV.
        json.dump({**row, "log_history": log}, open(json_path, "w"), default=str)
        pd.DataFrame([row]).to_csv(RESULTS_CSV, mode="a",
                                   header=not os.path.exists(RESULTS_CSV), index=False)

    del tr, m, base, opt
    gc.collect()
    if torch.cuda.is_available():
      torch.cuda.empty_cache()
    return row

In [ ]:
#Sanity Test
# Check: 3 non-empty groups, B's lr = 4 x A's lr, no crash.
test = run_roberta(lam=4, seed=42, epochs=3, stage="test", max_steps=50)
print(test["final_acc"], test["train_runtime_s"], test["peak_mem_gb"])

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [ ]:
# Step 1, SST-2 screening (1 seed, 3 epoch)
# Standard LoRA: both matrices share one lr, tried at three values.
# Step 1: Full 3-epoch hyperparameter selection
# Same seed for every configuration
for lr in [1e-4, 2e-4, 4e-4]:
    run_roberta(
        lam=1,
        seed=SEED_SCREEN,
        epochs=3,
        base_lr=lr,
        stage="screening")

# LoRA+: three ratios x two values of eta_A.
for lam in [4, 8, 16]:
    for lr in [1e-4, 2e-4]:
        run_roberta(
            lam=lam,
            seed=SEED_SCREEN,
            epochs=3,
            base_lr=lr,
            stage="screening")

NameError: name 'run_roberta' is not defined

In [ ]:
#Screening Results
import pandas as pd

df = pd.read_csv(RESULTS_CSV)

# Step 1: Filter completed 3-epoch screening runs
screening = df[
    (df["stage"] == "screening") &
    (df["epochs"] == 3)
].copy()

unique_runs = screening.drop_duplicates(
    subset=["method", "lam", "eta_A"]
)

screening = unique_runs.copy()

# Step 2: Verify all 9 experiments completed
print("Completed tuning runs:", len(screening))

assert len(screening) == 9, (
    f"Expected 9 screening runs, found {len(screening)}"
)

# Step 3: Sort by validation accuracy
screening = screening.sort_values(
    by=["final_acc", "final_eval_loss"],
    ascending=[False, True]
)

best_lora = screening[
    screening["method"] == "LoRA"
].iloc[0]

best_loraplus = screening[
    screening["method"] == "LoRA+"
].iloc[0]

print("Best LoRA:")
print(best_lora[
    ["eta_A", "final_acc", "final_f1_macro"]
])

print("\nBest LoRA+:")
print(best_loraplus[
    ["lam", "eta_A", "eta_B",
     "final_acc", "final_f1_macro"]
])

In [ ]:
# FILL THESE IN from Cell 12 before running.
# Step 2, SST-2 final (3 seeds, 3 epochs)
BEST_LORA_LR = float(best_lora["eta_A"])

BEST_LAM = int(best_loraplus["lam"])

BEST_LORAPLUS_LR = float(best_loraplus["eta_A"])

for seed in FINAL_SEEDS:
    run_roberta(lam=1, seed=seed, epochs=3, base_lr=BEST_LORA_LR, stage="final")
    run_roberta(lam=BEST_LAM, seed=seed, epochs=3, base_lr=BEST_LORAPLUS_LR, stage="final")

In [ ]:
#Results Table(mean ± std over seeds)
df = pd.read_csv(RESULTS_CSV)

fin = df[
    (df["stage"] == "final") &
    (df["epochs"] == 3)
]

summary = fin.groupby(
    ["method", "lam", "eta_A"]
)[
    [
        "final_acc",
        "final_f1_macro",
        "final_eval_loss",
        "train_runtime_s",
        "peak_mem_gb",
        "trainable_params"
    ]
].agg(["mean", "std"])

print(summary)

In [ ]:
# convergence curves + steps to target
runs = [json.load(open(f)) for f in glob.glob(f"{ROOT}/runs/*.json")]
runs = [r for r in runs if r["stage"] == "final" and r["dataset"] == DATASET]

# Group runs by method setting, then average the curves over seeds.
curves = {}
for r in runs:
    key = f'{r["method"]} (lam={r["lam"]}, etaA={r["eta_A"]})'
    pts = [(l["step"], l["eval_accuracy"]) for l in r["log_history"] if "eval_accuracy" in l]
    curves.setdefault(key, []).append(pts)

plt.figure(figsize=(8, 5))
means = {}
for key, group in curves.items():
    n = min(len(g) for g in group)
    steps = [s for s, _ in group[0][:n]]
    mean = np.mean([[a for _, a in g[:n]] for g in group], axis=0)
    means[key] = (steps, mean)
    plt.plot(steps, mean, label=key)
plt.xlabel("step"); plt.ylabel("validation accuracy"); plt.legend()
plt.savefig(f"{ROOT}/convergence_{DATASET}.png", dpi=150, bbox_inches="tight")
plt.show()

# Steps to reach a target accuracy (the paper's "speedup" idea).
TARGET = 0.93   # example: choose e.g. the baseline's mean final accuracy
for key, (steps, mean) in means.items():
    hit = next((s for s, a in zip(steps, mean) if a >= TARGET), None)
    print(key, "-> first step reaching", TARGET, ":", hit)

In [ ]:
# Cell 18: Print evaluation history from a saved final run

import glob
import json

files = sorted(glob.glob(f"{ROOT}/runs/*_final.json"))

if files:
    with open(files[0], "r") as f:
        result = json.load(f)

    print("Reading:", files[0])

    for log in result["log_history"]:
        if "eval_accuracy" in log:
            print(
                f"Epoch {log.get('epoch', 0):.2f}: "
                f"Eval Loss = {log.get('eval_loss', 0):.4f}, "
                f"Eval Accuracy = {log['eval_accuracy']:.4f}, "
                f"Macro-F1 = {log.get('eval_f1_macro', 0):.4f}"
            )
else:
    print("No completed final runs found.")

Epoch 1.0: Eval Loss = 0.1974, Eval Accuracy = 0.9346
Epoch 2.0: Eval Loss = 0.2138, Eval Accuracy = 0.9381
Epoch 3.0: Eval Loss = 0.2074, Eval Accuracy = 0.9335


In [ ]:
# Run ONLY after every JSON and the CSV are on Drive.
# from google.colab import runtime
# runtime.unassign()


---
# PART 3 — ASNEEM | GPT-2 + QQP

**Assigned to:** Asneem Athar Shaik  
**Model:** GPT-2  
**Dataset:** GLUE — Quora Question Pairs (QQP)  
**Experiment:** Standard LoRA vs. LoRA+

### Responsibilities
- GPT-2 fine-tuning on QQP
- LoRA parameter initialization and configuration
- LoRA+ learning-rate configuration
- Model checkpoint saving and backups
- Experimental setup and documentation

### Team Collaboration Note
> This section contains Asneem's experiments. Please avoid modifying the code, configurations, or checkpoint paths below without coordinating first. Other team members' experiments are maintained separately.




In [ ]:
import torch
import transformers
import peft
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

PyTorch: 2.14.1+cu130
Transformers: 5.22.1
PEFT: 0.21.2
Accelerate: 1.15.0


In [ ]:
import os
import json


from transformers import set_seed

SEED = 42
set_seed(SEED)

CONFIG = {
    "owner": "Asneem",
    "model_name": "gpt2",
    "dataset": "qqp",
    "max_length": 128,
    "rank": 8,
    "lora_alpha": 8,
    "lora_dropout": 0.0,
    "learning_rate": 1e-4,
    "lambda_ratios": [1, 4, 16],
    "epochs": 3,
    "seed": SEED,
    "pilot_train_samples": 2000,
    "pilot_eval_samples": 500,
}

BASE_DIR = "/content/drive/MyDrive/LoRA_Project/Asneem_GPT2_QQP"
os.makedirs(BASE_DIR, exist_ok=True)

with open(os.path.join(BASE_DIR, "config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print(json.dumps(CONFIG, indent=2))

{
  "owner": "Asneem",
  "model_name": "gpt2",
  "dataset": "qqp",
  "max_length": 128,
  "rank": 8,
  "lora_alpha": 8,
  "lora_dropout": 0.0,
  "learning_rate": 0.0001,
  "lambda_ratios": [
    1,
    4,
    16
  ],
  "epochs": 3,
  "seed": 42,
  "pilot_train_samples": 2000,
  "pilot_eval_samples": 500
}


In [ ]:
from datasets import load_dataset
qqp = load_dataset("nyu-mll/glue", "qqp")

print(qqp)
print(qqp["train"][0])

README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

qqp/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 33.6MB            

qqp/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

qqp/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.73MB            

qqp/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

qqp/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 36.7MB            

qqp/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 363846
    })
    validation: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 40430
    })
    test: Dataset({
        features: ['question1', 'question2', 'label', 'idx'],
        num_rows: 390965
    })
})
{'question1': 'How is the life of a math student? Could you describe your own experiences?', 'question2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0}


In [ ]:
train_dataset = (
    qqp["train"]
    .shuffle(seed=SEED)
    .select(range(CONFIG["pilot_train_samples"]))
)

eval_dataset = (
    qqp["validation"]
    .shuffle(seed=SEED)
    .select(range(CONFIG["pilot_eval_samples"]))
)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(eval_dataset))

Training samples: 2000
Validation samples: 500


In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"]
)

# GPT-2 has no default padding token.
tokenizer.pad_token = tokenizer.eos_token

def tokenize_qqp(batch):
    return tokenizer(
        batch["question1"],
        batch["question2"],
        truncation=True,
        max_length=CONFIG["max_length"],
        padding=False,
    )

train_tokenized = train_dataset.map(
    tokenize_qqp,
    batched=True,
    remove_columns=[
        "question1", "question2", "idx"
    ],
)

eval_tokenized = eval_dataset.map(
    tokenize_qqp,
    batched=True,
    remove_columns=[
        "question1", "question2", "idx"
    ],
)

print(train_tokenized[0].keys())
print(tokenizer.decode(train_tokenized[0]["input_ids"]))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

dict_keys(['label', 'input_ids', 'attention_mask'])
What would Rhaegar and Jon have thought of each other?What is the amount of torque a 2000 Jeep Cherokee can output?


In [ ]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType
base_model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    pad_token_id=tokenizer.pad_token_id,
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=CONFIG["rank"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=["c_attn"],
    fan_in_fan_out=True,
    modules_to_save=["score"],
    bias="none",
    init_lora_weights=True,
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

NameError: name 'CONFIG' is not defined

In [ ]:
from torch.optim import AdamW

def create_loraplus_optimizer(model, base_lr=1e-4, lambda_ratio=4):
    lora_A_params = []
    lora_B_params = []
    other_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if "lora_A" in name:
            lora_A_params.append(param)

        elif "lora_B" in name:
            lora_B_params.append(param)

        else:
            # Includes the GPT-2 classification head
            other_params.append(param)

    optimizer = AdamW(
        [
            {
                "params": lora_A_params,
                "lr": base_lr,
            },
            {
                "params": lora_B_params,
                "lr": base_lr * lambda_ratio,
            },
            {
                "params": other_params,
                "lr": base_lr,
            },
        ],
        weight_decay=0.01,
    )

    return optimizer

In [ ]:
optimizer = create_loraplus_optimizer(
    model,
    base_lr=CONFIG["learning_rate"],
    lambda_ratio=4,
)

group_names = ["LoRA A", "LoRA B", "Classification head"]

for name, group in zip(group_names, optimizer.param_groups):
    num_params = sum(p.numel() for p in group["params"])

    print(f"{name}")
    print(f"  Learning rate: {group['lr']}")
    print(f"  Parameters: {num_params:,}")

LoRA A
  Learning rate: 0.0001
  Parameters: 73,728
LoRA B
  Learning rate: 0.0004
  Parameters: 221,184
Classification head
  Learning rate: 0.0001
  Parameters: 1,536


In [ ]:
%pip install -q evaluate

In [ ]:
import evaluate
import numpy as np

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_metric.compute(
            predictions=predictions,
            references=labels
        )["accuracy"],
        "f1": f1_metric.compute(
            predictions=predictions,
            references=labels
        )["f1"],
    }

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=f"{BASE_DIR}/pilot_lora_baseline",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=CONFIG["learning_rate"],
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=25,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED,
    data_seed=SEED,
    remove_unused_columns=True,
    save_total_limit=2,
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
set_seed(SEED)

baseline_base = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=2,
    pad_token_id=tokenizer.pad_token_id,
)

baseline_model = get_peft_model(
    baseline_base,
    lora_config
)

baseline_optimizer = create_loraplus_optimizer(
    baseline_model,
    base_lr=CONFIG["learning_rate"],
    lambda_ratio=1,
)

baseline_model.print_trainable_parameters()

print("Standard LoRA baseline initialized!")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 296,448 || all params: 124,737,792 || trainable%: 0.2377
Standard LoRA baseline initialized!


In [ ]:
trainer = Trainer(
    model=baseline_model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    optimizers=(baseline_optimizer, None),
)

train_result = trainer.train()

eval_result = trainer.evaluate()

print("\nBASELINE RESULTS")
print("Accuracy:", eval_result["eval_accuracy"])
print("F1 Score:", eval_result["eval_f1"])
print("Training time:", train_result.metrics["train_runtime"], "seconds")

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.789705,0.667478,0.636000,0.354610


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.789705,0.667478,1,0.636000,0.354610



BASELINE RESULTS
Accuracy: 0.636
F1 Score: 0.3546099290780142
Training time: 31.6411 seconds


In [ ]:
import os
import json

baseline_dir = f"{BASE_DIR}/pilot_lora_baseline/final"
os.makedirs(baseline_dir, exist_ok=True)

trainer.save_model(baseline_dir)
tokenizer.save_pretrained(baseline_dir)

baseline_summary = {
    "model": "gpt2",
    "dataset": "qqp",
    "experiment": "Standard LoRA",
    "lambda_ratio": 1,
    "seed": SEED,
    "train_samples": len(train_tokenized),
    "eval_samples": len(eval_tokenized),
    "accuracy": float(eval_result["eval_accuracy"]),
    "f1": float(eval_result["eval_f1"]),
    "train_runtime": float(train_result.metrics["train_runtime"]),
}

with open(f"{baseline_dir}/results.json", "w") as f:
    json.dump(baseline_summary, f, indent=2)

print("Baseline saved:", baseline_dir)

Baseline saved: /content/drive/MyDrive/LoRA_Project/Asneem_GPT2_QQP/pilot_lora_baseline/final


In [ ]:
from transformers import Trainer, TrainingArguments

def run_loraplus_experiment(lambda_ratio):
    print(f"\nStarting LoRA+ experiment: lambda={lambda_ratio}")

    # Identical initialization for every experiment
    set_seed(SEED)

    fresh_base = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=2,
        pad_token_id=tokenizer.pad_token_id,
    )

    fresh_model = get_peft_model(fresh_base, lora_config)

    experiment_dir = (
        f"{BASE_DIR}/pilot_loraplus_lambda_{lambda_ratio}"
    )

    args = TrainingArguments(
        output_dir=experiment_dir,
        num_train_epochs=1,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=CONFIG["learning_rate"],
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=25,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
        save_total_limit=2,
    )

    optimizer = create_loraplus_optimizer(
        fresh_model,
        base_lr=CONFIG["learning_rate"],
        lambda_ratio=lambda_ratio,
    )

    experiment_trainer = Trainer(
        model=fresh_model,
        args=args,
        train_dataset=train_tokenized,
        eval_dataset=eval_tokenized,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        optimizers=(optimizer, None),
    )

    training_output = experiment_trainer.train()
    evaluation = experiment_trainer.evaluate()

    final_dir = f"{experiment_dir}/final"
    experiment_trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)

    summary = {
        "model": "gpt2",
        "dataset": "qqp",
        "lambda_ratio": lambda_ratio,
        "seed": SEED,
        "train_samples": len(train_tokenized),
        "eval_samples": len(eval_tokenized),
        "accuracy": float(evaluation["eval_accuracy"]),
        "f1": float(evaluation["eval_f1"]),
        "train_runtime": float(
            training_output.metrics["train_runtime"]
        ),
    }

    with open(f"{final_dir}/results.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nExperiment completed:")
    print(json.dumps(summary, indent=2))

    del experiment_trainer, fresh_model, fresh_base, optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

In [ ]:
result_lambda4 = run_loraplus_experiment(4)


Starting LoRA+ experiment: lambda=4


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.725135,0.619289,0.672000,0.464052


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.725135,0.619289,1,0.672000,0.464052



Experiment completed:
{
  "model": "gpt2",
  "dataset": "qqp",
  "lambda_ratio": 4,
  "seed": 42,
  "train_samples": 2000,
  "eval_samples": 500,
  "accuracy": 0.672,
  "f1": 0.46405228758169936,
  "train_runtime": 12.9374
}


In [ ]:
result_lambda16 = run_loraplus_experiment(16)


Starting LoRA+ experiment: lambda=16


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.947663,0.695081,0.636000,0.397351


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.947663,0.695081,1,0.636000,0.397351



Experiment completed:
{
  "model": "gpt2",
  "dataset": "qqp",
  "lambda_ratio": 16,
  "seed": 42,
  "train_samples": 2000,
  "eval_samples": 500,
  "accuracy": 0.636,
  "f1": 0.3973509933774834,
  "train_runtime": 19.0608
}


In [ ]:
import pandas as pd

results_df = pd.DataFrame([
    baseline_summary,
    result_lambda4,
    result_lambda16,
])

print(
    results_df[
        ["lambda_ratio", "accuracy", "f1", "train_runtime"]
    ].to_string(index=False)
)

 lambda_ratio  accuracy       f1  train_runtime
            1     0.636 0.354610        31.6411
            4     0.672 0.464052        12.9374
           16     0.636 0.397351        19.0608


In [ ]:
train_20k = (
    qqp["train"]
    .shuffle(seed=SEED)
    .select(range(20_000))
    .map(
        tokenize_qqp,
        batched=True,
        remove_columns=["question1", "question2", "idx"]
    )
)

eval_2k = (
    qqp["validation"]
    .shuffle(seed=SEED)
    .select(range(2_000))
    .map(
        tokenize_qqp,
        batched=True,
        remove_columns=["question1", "question2", "idx"]
    )
)

print("Training:", len(train_20k))
print("Validation:", len(eval_2k))

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Training: 20000
Validation: 2000


In [ ]:
def run_three_epoch_experiment(lambda_ratio):
    set_seed(SEED)

    experiment_dir = (
        f"{BASE_DIR}/three_epoch_lambda_{lambda_ratio}"
    )

    base = AutoModelForSequenceClassification.from_pretrained(
        CONFIG["model_name"],
        num_labels=2,
        pad_token_id=tokenizer.pad_token_id,
    )

    experiment_model = get_peft_model(base, lora_config)

    optimizer = create_loraplus_optimizer(
        experiment_model,
        base_lr=CONFIG["learning_rate"],
        lambda_ratio=lambda_ratio,
    )

    args = TrainingArguments(
        output_dir=experiment_dir,
        num_train_epochs=3,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=100,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
        save_total_limit=2,
    )

    experiment_trainer = Trainer(
        model=experiment_model,
        args=args,
        train_dataset=train_20k,
        eval_dataset=eval_2k,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        optimizers=(optimizer, None),
    )

    train_output = experiment_trainer.train()
    metrics = experiment_trainer.evaluate()

    final_dir = f"{experiment_dir}/final"
    experiment_trainer.save_model(final_dir)
    tokenizer.save_pretrained(final_dir)

    summary = {
        "lambda_ratio": lambda_ratio,
        "epochs": 3,
        "train_samples": len(train_20k),
        "eval_samples": len(eval_2k),
        "accuracy": float(metrics["eval_accuracy"]),
        "f1": float(metrics["eval_f1"]),
        "train_runtime": float(
            train_output.metrics["train_runtime"]
        ),
        "log_history": experiment_trainer.state.log_history,
    }

    with open(f"{final_dir}/results.json", "w") as f:
        json.dump(summary, f, indent=2)

    del experiment_trainer, experiment_model, base, optimizer

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return summary

In [ ]:
result_3epoch_lora = run_three_epoch_experiment(1)

print({
    k: v for k, v in result_3epoch_lora.items()
    if k != "log_history"
})

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.491136,0.472435,0.755000,0.667571
2,0.470264,0.448767,0.785500,0.721248
3,0.460771,0.442047,0.789000,0.719041


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.460771,0.442047,3,0.789000,0.719041


{'lambda_ratio': 1, 'epochs': 3, 'train_samples': 20000, 'eval_samples': 2000, 'accuracy': 0.789, 'f1': 0.7190412782956058, 'train_runtime': 358.2753}


In [ ]:
result_3epoch_loraplus4 = run_three_epoch_experiment(4)

print({
    k: v for k, v in result_3epoch_loraplus4.items()
    if k != "log_history"
})

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.456141,0.472118,0.764500,0.723429
2,0.451327,0.432228,0.792000,0.734015
3,0.450672,0.431158,0.802500,0.740984


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.450672,0.431158,3,0.802500,0.740984


{'lambda_ratio': 4, 'epochs': 3, 'train_samples': 20000, 'eval_samples': 2000, 'accuracy': 0.8025, 'f1': 0.740983606557377, 'train_runtime': 396.9047}


In [ ]:
result_3epoch_loraplus16 = run_three_epoch_experiment(16)

print({
    k: v for k, v in result_3epoch_loraplus16.items()
    if k != "log_history"
})

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2ForSequenceClassification LOAD REPORT from: gpt2
Key          | Status  | 
-------------+---------+-
score.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.424041,0.460094,0.778000,0.733493
2,0.445642,0.418295,0.791000,0.737107
3,0.423778,0.424618,0.809000,0.749672


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.423778,0.424618,3,0.809000,0.749672


{'lambda_ratio': 16, 'epochs': 3, 'train_samples': 20000, 'eval_samples': 2000, 'accuracy': 0.809, 'f1': 0.7496723460026212, 'train_runtime': 385.7055}


In [ ]:
import pandas as pd
import os

results = [
    result_3epoch_lora,
    result_3epoch_loraplus4,
    result_3epoch_loraplus16,
]

comparison = pd.DataFrame([
    {
        "lambda_ratio": r["lambda_ratio"],
        "accuracy": r["accuracy"],
        "f1": r["f1"],
        "train_runtime_seconds": r["train_runtime"],
        "epochs": r["epochs"],
        "train_samples": r["train_samples"],
        "eval_samples": r["eval_samples"],
    }
    for r in results
])

comparison_path = os.path.join(
    BASE_DIR, "three_epoch_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)

display(comparison)
print("Saved to:", comparison_path)

,lambda_ratio,accuracy,f1,train_runtime_seconds,epochs,train_samples,eval_samples
0,1,0.7890,0.719041,358.2753,3,20000,2000
1,4,0.8025,0.740984,396.9047,3,20000,2000
2,16,0.8090,0.749672,385.7055,3,20000,2000


Saved to: /content/drive/MyDrive/LoRA_Project/Asneem_GPT2_QQP/three_epoch_comparison.csv
